# KIRA Chili Disease ML (Khusus Cabai)

Notebook ini **hanya** memakai dataset cabai `dhenyd/chili-plant-disease`.
Dataset tomat **tidak dipakai** karena labelnya rancu dan membuat model menebak
tanaman lain sebagai tomat.

Kelas yang dipakai:

1. `healthy`
2. `leaf curl`
3. `leaf spot`
4. `whitefly`
5. `yellowish`

Output: `kira_ml_model.joblib` yang berisi model klasifikasi + `LabelEncoder`.

In [ ]:
!pip -q install xgboost scikit-learn timm torch torchvision matplotlib seaborn pillow joblib

In [ ]:
import glob
import os
import random
from collections import Counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import timm
import torch
import xgboost as xgb
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torchvision import transforms

SEED = 42
IMG_SIZE = 192
MODEL_NAME = "mobilenetv3_large_100"
AUG_PER_IMAGE = 4


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Load dataset cabai

Cari folder dataset di beberapa lokasi Kaggle yang umum. Hanya folder cabai
yang dipakai.

In [ ]:
CANDIDATES = [
    "/kaggle/input/chili-plant-disease",
    "/kaggle/input/datasets/dhenyd/chili-plant-disease",
    "/kaggle/input/chili-plant-disease/chili-plant-disease",
]

DATA_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
if DATA_DIR is None:
    raise FileNotFoundError("Dataset cabai tidak ditemukan. Cek nama dataset di Kaggle.")

print("DATA_DIR:", DATA_DIR)
print("isi:", sorted(os.listdir(DATA_DIR)))

In [ ]:
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")
SPLIT_ALIASES = {
    "train": ["train"],
    "val": ["val", "valid", "validation"],
    "test": ["test"],
}


def find_split_dir(data_dir, split):
    for name in SPLIT_ALIASES[split]:
        path = os.path.join(data_dir, name)
        if os.path.isdir(path):
            return path
    return None


def collect_split(data_dir, split):
    root = find_split_dir(data_dir, split)
    if root is None:
        return [], []

    paths, labels = [], []
    for class_name in sorted(os.listdir(root)):
        class_dir = os.path.join(root, class_name)
        if not os.path.isdir(class_dir):
            continue
        if class_name.lower() in {"train", "val", "valid", "validation", "test"}:
            continue

        images = []
        for ext in IMAGE_EXTS:
            images.extend(glob.glob(os.path.join(class_dir, "**", f"*{ext}"), recursive=True))
            images.extend(glob.glob(os.path.join(class_dir, "**", f"*{ext.upper()}"), recursive=True))
        images = sorted(set(images))

        label = f"chili_{class_name.strip().replace(' ', '_').lower()}"
        for path in images:
            paths.append(path)
            labels.append(label)

    return paths, labels


train_paths, train_labels = collect_split(DATA_DIR, "train")
val_paths, val_labels = collect_split(DATA_DIR, "val")
test_paths, test_labels = collect_split(DATA_DIR, "test")

print("train:", len(train_paths))
print("val  :", len(val_paths))
print("test :", len(test_paths))
print("kelas:", sorted(set(train_labels)))

assert train_paths, "Tidak ada gambar train."
assert all("tomato" not in label.lower() for label in train_labels), "Dataset tomat tidak boleh dipakai."

In [ ]:
# EDA distribusi kelas
all_counts = Counter(train_labels + val_labels + test_labels)
plt.figure(figsize=(8, 4))
plt.bar(all_counts.keys(), all_counts.values(), color="#d4a373")
plt.xticks(rotation=25, ha="right")
plt.title("Distribusi Kelas Cabai")
plt.tight_layout()
plt.show()

In [ ]:
# Contoh gambar train
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, path, label in zip(axes.flatten(), train_paths[:10], train_labels[:10]):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(label.replace("chili_", ""), fontsize=9)
    ax.axis("off")
plt.suptitle("Contoh Gambar Daun Cabai")
plt.tight_layout()
plt.show()

## 2. Ekstraksi fitur MobileNetV3

Preprocessing harus sama persis dengan `kira-ml-service/main.py`:
`Resize(192,192) -> ToTensor -> Normalize(ImageNet)`.

In [ ]:
base_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

aug_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

extractor = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0).to(device).eval()

In [ ]:
def embed_paths(paths, tf, batch_size=32):
    feats = []
    batch = []

    def flush():
        if not batch:
            return
        tensor = torch.stack(batch).to(device)
        with torch.no_grad():
            feats.append(extractor(tensor).cpu().numpy())
        batch.clear()

    for path in paths:
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            continue
        batch.append(tf(image))
        if len(batch) >= batch_size:
            flush()
    flush()
    return np.concatenate(feats, axis=0) if feats else np.empty((0, 0), dtype=np.float32)


print("Ekstraksi train (asli)...")
x_train = embed_paths(train_paths, base_tf)

print(f"Ekstraksi train ({AUG_PER_IMAGE}x augmentasi)...")
x_aug = embed_paths(train_paths * AUG_PER_IMAGE, aug_tf)

print("Ekstraksi val...")
x_val = embed_paths(val_paths, base_tf)

print("Ekstraksi test...")
x_test = embed_paths(test_paths, base_tf)

y_train = np.array(train_labels)
y_val = np.array(val_labels)
y_test = np.array(test_labels)

print("shape:", x_train.shape, x_aug.shape, x_val.shape, x_test.shape)

## 3. Training model

LabelEncoder hanya untuk 5 kelas cabai. Model yang dipilih adalah yang paling
akurat di validation set.

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test)

x_train_fit = np.concatenate([x_train, x_aug], axis=0)
y_train_fit = np.concatenate([y_train_enc, np.tile(y_train_enc, AUG_PER_IMAGE)])

print("kelas:", list(le.classes_))
print("shape train fit:", x_train_fit.shape)

In [ ]:
try:
    xgb_model = xgb.XGBClassifier(
        n_estimators=700,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED,
        early_stopping_rounds=60,
    )
    xgb_model.fit(x_train_fit, y_train_fit, eval_set=[(x_val, y_val_enc)], verbose=False)
except TypeError:
    xgb_model = xgb.XGBClassifier(
        n_estimators=700,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED,
    )
    xgb_model.fit(x_train_fit, y_train_fit, verbose=False)

logreg_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, C=1.0, n_jobs=-1, random_state=SEED),
)
logreg_model.fit(x_train_fit, y_train_fit)

models = {"xgboost": xgb_model, "logreg": logreg_model}
val_scores = {}
for name, model in models.items():
    pred = model.predict(x_val)
    val_scores[name] = accuracy_score(y_val_enc, pred)
    print(f"{name} val accuracy: {val_scores[name]:.4f}")

best_name = max(val_scores, key=val_scores.get)
best_model = models[best_name]
print("model terpilih:", best_name)

In [ ]:
# Evaluasi final di test set
pred_test = best_model.predict(x_test)
print("test accuracy:", accuracy_score(y_test_enc, pred_test))
print(classification_report(y_test_enc, pred_test, target_names=le.classes_, digits=3, zero_division=0))

cm = confusion_matrix(y_test_enc, pred_test)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title(f"Confusion Matrix - {best_name}")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Simpan model

Key `model` dipakai `kira-ml-service/main.py`. Key `xgb` tetap disimpan untuk
kompatibilitas bundle lama.

In [ ]:
out_path = "/kaggle/working/kira_ml_model.joblib"
bundle = {
    "model": best_model,
    "xgb": xgb_model,
    "le": le,
    "classes": list(le.classes_),
}
joblib.dump(bundle, out_path, compress=3)
print("disimpan:", out_path)
print("classes:", bundle["classes"])

In [ ]:
# Sanity check: load ulang dan prediksi satu gambar test
loaded = joblib.load("/kaggle/working/kira_ml_model.joblib")
loaded_model = loaded["model"]
loaded_le = loaded["le"]

sample_path = test_paths[0]
sample_img = Image.open(sample_path).convert("RGB")
feat = embed_paths([sample_path], base_tf)
pred = int(np.argmax(loaded_model.predict_proba(feat)[0]))
print("file     :", sample_path)
print("aktual   :", test_labels[0])
print("prediksi :", loaded_le.classes_[pred])